# TopoSurface-DTI — Kaggle Notebook (Real PDBBind Data)

Trains the full TopoSurface-DTI pipeline on real PDBBind protein-ligand binding affinity data.

**Before running, attach two datasets to this notebook:**

1. **Code dataset:** `drug-target-gdl`  
   (Add Data → Your Datasets → drug-target-gdl)

2. **PDBBind dataset:** `pdbbind-protein-ligand-binding-affinity-dataset` by madukacharles  
   (Add Data → Search `pdbbind-protein-ligand-binding-affinity-dataset` → madukacharles)

In [ ]:
# ── 1a. Install basic dependencies ───────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ripser', 'pyyaml'])
print('Basic dependencies installed.')

In [ ]:
# ── 1b. Install real-data dependencies ───────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'biopython', 'rdkit'])
print('Real-data dependencies installed.')

In [ ]:
# ── 2. Copy project code to /kaggle/working (writable) ───────────────────
import shutil, os, sys

repo_path = '/kaggle/working/Drug_Target_GDL'

# 1. Erase the old code completely
if os.path.exists(repo_path):
    shutil.rmtree(repo_path)
    print("Old repository deleted.")

# 2. Clone the fresh code from GitHub
!git clone https://github.com/aumrawal/Drug_Target_GDL-TDA.git {repo_path}
print("New code successfully cloned!")

# 3. Add to Python path and change working directory
sys.path.insert(0, repo_path)
os.chdir(repo_path)
print(f'Working directory set to: {os.getcwd()}')

In [ ]:
# ── 3. Merge PDBBind data into /kaggle/working/pdbbind (flat view) ────────
# Handles PDBbind-v2020 layout where PDB IDs live under year-range subdirs:
#   P-L/1981-2000/{pdb_id}/   P-L/2001-2010/{pdb_id}/   P-L/2011-2019/{pdb_id}/
import os, shutil

PDBBIND_FLAT = '/kaggle/working/pdbbind'

# Clean up any previous link/directory
if os.path.lexists(PDBBIND_FLAT):
    if os.path.islink(PDBBIND_FLAT) or os.path.isfile(PDBBIND_FLAT):
        os.remove(PDBBIND_FLAT)
    elif os.path.isdir(PDBBIND_FLAT):
        shutil.rmtree(PDBBIND_FLAT)
os.makedirs(PDBBIND_FLAT)

def merge_pdb_dirs(dest, search_root='/kaggle/input', max_depth=8):
    """
    Walk search_root, find directories whose names are 4-char alphanumeric PDB IDs,
    and symlink each into dest to produce a flat merged view.
    Does NOT recurse into PDB-ID directories themselves.
    """
    count = 0
    for root, dirs, _ in os.walk(search_root):
        rel_depth = root.replace(search_root, '').count(os.sep)
        if rel_depth > max_depth:
            del dirs[:]
            continue
        pdb_ids = [d for d in dirs if len(d) == 4 and d.isalnum()]
        for pdb_id in pdb_ids:
            src = os.path.join(root, pdb_id)
            dst = os.path.join(dest, pdb_id)
            if not os.path.lexists(dst):
                os.symlink(src, dst)
                count += 1
        dirs[:] = [d for d in dirs if d not in pdb_ids]  # don't recurse into PDB dirs
    return count

n_linked = merge_pdb_dirs(PDBBIND_FLAT)
all_entries = sorted(os.listdir(PDBBIND_FLAT))
print(f'Merged {n_linked} PDB ID directories → {PDBBIND_FLAT}')
print(f'Total PDB entries: {len(all_entries)}, first 5: {all_entries[:5]}')

# ── Affinity index file: use the Protein-Ligand index from the index dataset ──
# The 4 index files are NL / PL / PN / PP — we want PL (protein-ligand = DTI).
_PL_INDEX = '/kaggle/input/datasets/rakeshrawal87/index-files/index/INDEX_general_PL.2020R1.lst'

if os.path.isfile(_PL_INDEX):
    INDEX_FILE = _PL_INDEX
    print(f'Index file (PL): {INDEX_FILE}')
else:
    # Fallback: search non-PDB directories under /kaggle/input for any .lst / .csv / .txt
    INDEX_FILE = None
    _skip_exts = {'.mol2', '.pdb', '.sdf', '.cif', '.mol', '.ent', '.pdbqt'}
    for _root, _dirs, _files in os.walk('/kaggle/input'):
        _dirs[:] = [d for d in _dirs if not (len(d) == 4 and d.isalnum())]
        for _f in sorted(_files):
            if os.path.splitext(_f)[1].lower() not in _skip_exts:
                INDEX_FILE = os.path.join(_root, _f)
                break
        if INDEX_FILE:
            break
    print(f'PL index not found at expected path — fallback index: {INDEX_FILE}')

In [ ]:
# ── Diagnostic: find any non-structure file in the entire dataset ─────────
import os

DATASET_ROOT = '/kaggle/input/datasets/rakeshrawal87/pdbbind-v2020'
STRUCTURE_EXTS = {'.mol2', '.pdb', '.sdf', '.cif', '.mol', '.ent', '.pdbqt'}

# ── 1. Print the full directory tree above PDB-ID level ───────────────────
print('=== Directory tree (above PDB-ID level) ===')
for root, dirs, files in os.walk(DATASET_ROOT):
    # Compute indent from depth relative to root
    depth  = root.replace(DATASET_ROOT, '').count(os.sep)
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root)}/')
    for f in sorted(files):
        print(f'{indent}  {f}')
    # Stop recursing into 4-char alphanumeric dirs (PDB IDs)
    dirs[:] = [d for d in dirs if not (len(d) == 4 and d.isalnum())]

# ── 2. Collect every non-structure file anywhere (including inside PDB dirs) ──
print('\n=== Non-structure files found anywhere in dataset ===')
non_struct = []
for root, dirs, files in os.walk(DATASET_ROOT):
    for f in files:
        if os.path.splitext(f)[1].lower() not in STRUCTURE_EXTS:
            non_struct.append(os.path.join(root, f))

if non_struct:
    for p in non_struct[:20]:   # cap at 20 lines
        print(f'  {p}')
    if len(non_struct) > 20:
        print(f'  ... ({len(non_struct)} total)')
else:
    print('  (none — dataset contains only structure files)')

# ── 3. List all files in 3 sample PDB entries (one per year dir) ──────────
print('\n=== Sample PDB entry contents (one per year range) ===')
YEAR_DIRS = ['1981-2000', '2001-2010', '2011-2019']
for yr in YEAR_DIRS:
    yr_path = os.path.join(DATASET_ROOT, 'P-L', yr)
    try:
        pdb_ids = sorted(e for e in os.listdir(yr_path)
                         if len(e) == 4 and e.isalnum())
    except FileNotFoundError:
        print(f'  {yr}: directory not found'); continue
    if not pdb_ids:
        print(f'  {yr}: no PDB ID dirs found'); continue
    sample_id   = pdb_ids[0]
    sample_path = os.path.join(yr_path, sample_id)
    files       = sorted(os.listdir(sample_path))
    print(f'\n  {yr}/{sample_id}/:')
    for f in files:
        size = os.path.getsize(os.path.join(sample_path, f))
        print(f'    {f}  ({size:,} bytes)')

In [ ]:
# ── 4. Verify environment ─────────────────────────────────────────────────
import torch, numpy as np
from ripser import ripser

print(f'PyTorch : {torch.__version__}')
print(f'NumPy   : {np.__version__}')
print(f'GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU run)"}')

pts  = np.random.default_rng(0).uniform(0, 5, (10, 3))
dgms = ripser(pts, maxdim=1)['dgms']
print(f'Ripser  : OK  (H0={len(dgms[0])} bars, H1={len(dgms[1])} bars)')

In [ ]:
# ── 5. Generate train/val/test split JSON files ───────────────────────────
import sys, os, subprocess, json

# Re-establish path and working directory
repo_path = '/kaggle/working/Drug_Target_GDL'
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
os.chdir(repo_path)

DATA_DIR  = '/kaggle/working/pdbbind'
SPLIT_DIR = '/kaggle/working'

cmd = [sys.executable, 'scripts/make_splits.py',
       '--data_dir', DATA_DIR,
       '--out_dir',  SPLIT_DIR]

# INDEX_FILE was detected in cell 3; pass it explicitly so make_splits.py
# doesn't search the flat symlink dir (which contains only PDB subdirs, no index).
if 'INDEX_FILE' in dir() and INDEX_FILE:
    cmd += ['--index_file', INDEX_FILE]
    print(f'Using index file: {INDEX_FILE}')
else:
    print('No explicit index file detected — make_splits.py will search data_dir.')

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('Split generation failed — check the output above.')

# Verify splits were created
for split in ['train', 'val', 'test']:
    path = os.path.join(SPLIT_DIR, f'{split}_split.json')
    with open(path) as f:
        data = json.load(f)
    print(f'{split:5s}: {len(data)} samples  →  {path}')

In [ ]:
# ── 6. Train on real PDBBind data ─────────────────────────────────────────
import sys, os

repo_path = '/kaggle/working/Drug_Target_GDL'
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
os.chdir(repo_path)

import yaml, torch, torch.nn as nn, numpy as np
from torch.utils.data import DataLoader
from data.dataset           import DTIDataset
from train.trainer          import collate_single, train_epoch, validate
from models.toposurface_dti import TopoSurfaceDTI

DATA_DIR  = '/kaggle/working/pdbbind'
SPLIT_DIR = '/kaggle/working'

with open('configs/base.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['use_synthetic'] = False
cfg['n_epochs']      = 15   # ~30-40 min/epoch using pocket PDB; fits in a Kaggle session
cfg['lr']            = 1e-3

class PDBBindDataset(DTIDataset):
    def __init__(self, structure_dir, split_dir, split, **kwargs):
        self.structure_dir = structure_dir
        super().__init__(data_dir=split_dir, split=split, use_synthetic=False, **kwargs)

    def _load_real(self, pdb_id):
        from data.molecule_graph import mol_to_graph, synthetic_drug_graph
        from data.pocket_mesh    import (extract_pocket_atoms, build_knn_graph,
                                         precompute_pocket_geometry, synthetic_pocket_graph)
        # ── Drug: try .sdf first, fall back to .mol2 ──────────────────────
        try:
            from rdkit import Chem
            sdf  = os.path.join(self.structure_dir, pdb_id, f'{pdb_id}_ligand.sdf')
            mol2 = os.path.join(self.structure_dir, pdb_id, f'{pdb_id}_ligand.mol2')
            if os.path.exists(sdf):
                mol = Chem.SDMolSupplier(sdf, removeHs=True)[0]
            elif os.path.exists(mol2):
                mol = Chem.MolFromMol2File(mol2, removeHs=True)
            else:
                raise FileNotFoundError(f'No ligand file (.sdf/.mol2) for {pdb_id}')
            drug = mol_to_graph(mol)
        except Exception as e:
            print(f'[warn] Drug {pdb_id}: {e}')
            drug = synthetic_drug_graph()

        # ── Pocket: use pre-extracted pocket PDB (~100KB) not full protein ─
        # _pocket.pdb is already filtered to binding-site residues, so
        # BioPython parses it ~5x faster than the full _protein.pdb.
        try:
            pdb  = os.path.join(self.structure_dir, pdb_id, f'{pdb_id}_pocket.pdb')
            sdf  = os.path.join(self.structure_dir, pdb_id, f'{pdb_id}_ligand.sdf')
            mol2 = os.path.join(self.structure_dir, pdb_id, f'{pdb_id}_ligand.mol2')
            lig_path = sdf if os.path.exists(sdf) else (mol2 if os.path.exists(mol2) else None)
            pos_np, feat_np = extract_pocket_atoms(
                pdb, cutoff_angstrom=self.pocket_cutoff,
                sdf_path=lig_path,
            )
            pos_t  = torch.from_numpy(pos_np)
            feat_t = torch.from_numpy(feat_np)
            ei     = torch.from_numpy(build_knn_graph(pos_np, k=self.knn_k))
            geo    = precompute_pocket_geometry(pos_t, ei)
            pocket = {'x': feat_t, 'pos': pos_t, 'edge_index': ei, **geo}
        except Exception as e:
            print(f'[warn] Pocket {pdb_id}: {e}')
            pocket = synthetic_pocket_graph()

        return drug, pocket

train_ds = PDBBindDataset(DATA_DIR, SPLIT_DIR, 'train', tda_resolution=cfg['tda_resolution'])
val_ds   = PDBBindDataset(DATA_DIR, SPLIT_DIR, 'val',   tda_resolution=cfg['tda_resolution'])
print(f'Train samples: {len(train_ds)} | Val samples: {len(val_ds)}')

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,  collate_fn=collate_single)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False, collate_fn=collate_single)

train_affinities = torch.tensor([train_ds[i]['affinity'].item() for i in range(len(train_ds))])
target_mean = float(train_affinities.mean())
target_std  = float(train_affinities.std().clamp(min=1e-3))
print(f'Target  mean={target_mean:.3f}  std={target_std:.3f}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

model     = TopoSurfaceDTI.from_config(cfg).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=30, min_lr=1e-5)
loss_fn   = nn.MSELoss()

os.makedirs('checkpoints', exist_ok=True)
best_val_rmse = float('inf')
accum_steps   = cfg.get('accum_steps', 16)

for epoch in range(cfg['n_epochs']):
    tr = train_epoch(model, train_loader, optimizer, device, loss_fn, target_mean, target_std, accum_steps)
    va = validate(model, val_loader, device, loss_fn, target_mean, target_std)
    scheduler.step(va['loss'])
    print(f"Epoch {epoch:03d} | Train loss={tr['loss']:.4f} RMSE={tr['rmse']:.3f} R={tr['r']:.3f} | "
          f"Val loss={va['loss']:.4f} RMSE={va['rmse']:.3f} R={va['r']:.3f}")
    if va['rmse'] < best_val_rmse:
        best_val_rmse = va['rmse']
        torch.save({
            'epoch':         epoch,
            'model':         model.state_dict(),
            'optimizer':     optimizer.state_dict(),
            'scheduler':     scheduler.state_dict(),
            'best_val_rmse': best_val_rmse,
            'cfg':           cfg,
            'target_mean':   target_mean,
            'target_std':    target_std,
        }, 'checkpoints/best_model.pt')

print(f'\nTraining complete. Best Val RMSE: {best_val_rmse:.4f}')

In [ ]:
# ── 7. Evaluate on validation set ────────────────────────────────────────
import numpy as np
import torch
from scipy import stats
from torch.utils.data import DataLoader
from train.trainer import collate_single, forward_step

# Load target normalisation stats saved during training
ckpt        = torch.load('checkpoints/best_model.pt', map_location='cpu')
target_mean = ckpt.get('target_mean', 0.0)
target_std  = ckpt.get('target_std',  1.0)

device = next(model.parameters()).device
model.eval()
loss_fn = torch.nn.MSELoss()

loader = DataLoader(val_ds, batch_size=1, shuffle=False, collate_fn=collate_single)

preds, actuals = [], []
with torch.no_grad():
    for sample in loader:
        p, _ = forward_step(model, sample, device, loss_fn, target_mean, target_std)
        preds.append(float(p.cpu()))
        actuals.append(float(sample['affinity']))

preds   = np.array(preds)
actuals = np.array(actuals)
errors  = preds - actuals
abs_err = np.abs(errors)

rmse_val    = float(np.sqrt(np.mean(errors**2)))
mae         = float(np.mean(abs_err))
pearson_r,  _ = stats.pearsonr(preds, actuals)
spearman_r, _ = stats.spearmanr(preds, actuals)
bias        = float(np.mean(errors))
std_err     = float(np.std(errors))

print(f'n={len(preds)}  RMSE={rmse_val:.3f}  MAE={mae:.3f}  R={pearson_r:.3f}  ρ={spearman_r:.3f}')

In [ ]:
# ── 8. 4-panel visualisation ──────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.style.use('seaborn-v0_8-whitegrid')
fig = plt.figure(figsize=(14, 11))
fig.suptitle(
    f'TopoSurface-DTI  ·  Predicted vs Actual Binding Affinity  ·  n={len(preds)} compounds',
    fontsize=13, fontweight='bold', y=0.99
)
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32,
                       left=0.07, right=0.96, top=0.93, bottom=0.07)

# ── Panel 1: Scatter ─────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
sc  = ax1.scatter(actuals, preds, c=abs_err, cmap='RdYlGn_r', s=40,
                  alpha=0.75, edgecolors='none',
                  vmin=0, vmax=np.percentile(abs_err, 95))
lo  = min(actuals.min(), preds.min()) - 0.3
hi  = max(actuals.max(), preds.max()) + 0.3
ax1.plot([lo, hi], [lo, hi], 'k--', lw=1.2, label='Perfect prediction')
ax1.fill_between([lo, hi], [lo-rmse_val, hi-rmse_val], [lo+rmse_val, hi+rmse_val],
                 color='steelblue', alpha=0.08, label=f'±RMSE band')
slope, intercept, *_ = stats.linregress(actuals, preds)
xs = np.linspace(lo, hi, 100)
ax1.plot(xs, slope*xs + intercept, 'steelblue', lw=1.5, alpha=0.8, label='Linear fit')
plt.colorbar(sc, ax=ax1, label='|Error| (pKd)', shrink=0.85)
ax1.set(xlim=(lo,hi), ylim=(lo,hi),
        xlabel='Actual pKd', ylabel='Predicted pKd')
ax1.legend(fontsize=8, loc='upper left')
ax1.text(0.97, 0.05,
         f'RMSE = {rmse_val:.3f}\nMAE  = {mae:.3f}\nR    = {pearson_r:.3f}\nρ    = {spearman_r:.3f}',
         transform=ax1.transAxes, fontsize=9, va='bottom', ha='right',
         bbox=dict(boxstyle='round,pad=0.4', fc='white', alpha=0.85, ec='#cccccc'))
ax1.set_title('Predicted vs Actual', fontsize=12, fontweight='bold')

# ── Panel 2: Residuals histogram ──────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
n_bins = max(10, len(errors) // 8)
ax2.hist(errors, bins=n_bins, color='steelblue', alpha=0.65,
         edgecolor='white', linewidth=0.5, density=True, label='Residuals')
xs_r = np.linspace(errors.min()-0.5, errors.max()+0.5, 200)
ax2.plot(xs_r, stats.norm.pdf(xs_r, bias, std_err), 'crimson', lw=2,
         label=f'N({bias:.2f}, {std_err:.2f}²)')
ax2.axvline(0,        color='black',   lw=1.2, ls='--', alpha=0.7)
ax2.axvline(bias,     color='crimson', lw=1.2, ls=':',  alpha=0.9, label=f'Bias={bias:.3f}')
ax2.axvline( std_err, color='grey',    lw=0.8, ls=':',  alpha=0.6)
ax2.axvline(-std_err, color='grey',    lw=0.8, ls=':',  alpha=0.6)
within_1 = float(np.mean(abs_err < 1.0) * 100)
ax2.text(0.97, 0.95, f'{within_1:.1f}% within ±1 pKd',
         transform=ax2.transAxes, fontsize=9, va='top', ha='right',
         bbox=dict(boxstyle='round,pad=0.4', fc='white', alpha=0.85, ec='#cccccc'))
ax2.set(xlabel='Predicted − Actual (pKd)', ylabel='Density')
ax2.legend(fontsize=8)
ax2.set_title('Residuals Distribution', fontsize=12, fontweight='bold')

# ── Panel 3: Ranking view ─────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
order        = np.argsort(actuals)
xs_rank      = np.arange(len(order))
act_sorted   = actuals[order]
pred_sorted  = preds[order]
ax3.plot(xs_rank, act_sorted,  color='#2c7bb6', lw=2,   label='Actual pKd')
ax3.plot(xs_rank, pred_sorted, color='#d7191c', lw=1.5, alpha=0.8, label='Predicted pKd')
ax3.fill_between(xs_rank, act_sorted, pred_sorted,
                 where=(pred_sorted >= act_sorted),
                 color='#d7191c', alpha=0.12, label='Over-prediction')
ax3.fill_between(xs_rank, act_sorted, pred_sorted,
                 where=(pred_sorted <  act_sorted),
                 color='#2c7bb6', alpha=0.12, label='Under-prediction')
ax3.set(xlabel='Compound rank (sorted by actual pKd)', ylabel='pKd',
        xlim=(0, len(xs_rank)-1))
ax3.legend(fontsize=8, ncol=2)
ax3.set_title('Ranking View  (Virtual Screening)', fontsize=12, fontweight='bold')

# ── Panel 4: Cumulative error distribution ────────────────────────────────
ax4  = fig.add_subplot(gs[1, 1])
ae_s = np.sort(abs_err)
cdf  = np.arange(1, len(ae_s)+1) / len(ae_s)
ax4.plot(ae_s, cdf*100, color='#1a9641', lw=2.5)
ax4.fill_between(ae_s, cdf*100, alpha=0.15, color='#1a9641')
for t, lbl in [(0.5,'0.5'), (1.0,'1.0'), (1.5,'1.5'), (2.0,'2.0')]:
    pct = float(np.mean(ae_s <= t)*100)
    ax4.axvline(t, color='grey', lw=0.8, ls='--', alpha=0.6)
    ax4.text(t+0.02, 5, f'{pct:.0f}%\n≤{lbl}', fontsize=7.5, color='grey', va='bottom')
ax4.set(xlabel='Absolute error (pKd)', ylabel='Cumulative % of predictions',
        xlim=(0, None), ylim=(0, 102))
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0f}%'))
ax4.grid(axis='y', alpha=0.3)
ax4.set_title('Cumulative Error Distribution', fontsize=12, fontweight='bold')

plt.savefig('/kaggle/working/predictions_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to /kaggle/working/predictions_vs_actual.png')

print('\n── Evaluation Metrics ──────────────────────')
print(f'  RMSE         : {rmse_val:.4f} pKd')
print(f'  MAE          : {mae:.4f} pKd')
print(f'  Pearson R    : {pearson_r:.4f}')
print(f'  Spearman ρ   : {spearman_r:.4f}')
print(f'  Bias         : {bias:.4f} pKd')
print(f'  Error std    : {std_err:.4f} pKd')
for t in [0.5, 1.0, 1.5, 2.0]:
    print(f'  Within ±{t:.1f}   : {np.mean(abs_err<=t)*100:.1f}%')
print('────────────────────────────────────────────')